<a href="https://colab.research.google.com/github/khalidashani/Python/blob/main/Email_Domain_Checking_using_email_validator_library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

How to use email-validator to validate if the domain is valid or not

In [4]:
# Import all of the neccesary library to run the python code
import pandas as pd
import numpy as np
import sys
import subprocess
import dns.resolver
import matplotlib.pyplot as plt

from email_validator import validate_email, EmailNotValidError

In [9]:
# Data containing a mix of valid and invalid emails
data = {
    'User ID': range(101, 121),
    'email': [
        'khalid.pro@example.com',      # Valid
        'user123@gmail.com',           # Valid
        'invalid-email.com',           # Invalid: Missing @
        'test.user@company.co.uk',     # Valid: Multi-part TLD
        'admin@sub.domain.org',        # Valid
        'wrong@format@domain.com',     # Invalid: Multiple @
        'space in@email.com',          # Invalid: Space
        'hello@world.net',             # Valid
        'missing-tld@domain',          # Invalid: Missing .com/net/etc
        '@no-prefix.com',              # Invalid: No local part
        'valid.name+filter@gmail.com', # Valid: Plus tagging
        'double..dot@example.com',     # Invalid: Consecutive dots
        'user_name_12@provider.biz',   # Valid
        'special#char@domain.com',     # Invalid: Illegal character #
        'support@website.io',          # Valid
        'email@123.123.123.123',       # Valid: IP address literal
        '.start-dot@email.com',        # Invalid: Starts with dot
        'end-dot.@email.com',          # Invalid: Ends with dot in local
        'clean.email@edu.gov',         # Valid
        '  leading-space@test.com'      # Invalid: Leading spaces
    ]
}

# Create the DataFrame
df = pd.DataFrame(data)

# Display the first few rows
print(f"DataFrame Created with {len(df)} rows.")
df.head(10)

DataFrame Created with 20 rows.


,User ID,email
0,101,khalid.pro@example.com
1,102,user123@gmail.com
2,103,invalid-email.com
3,104,test.user@company.co.uk
4,105,admin@sub.domain.org
5,106,wrong@format@domain.com
6,107,space in@email.com
7,108,hello@world.net
8,109,missing-tld@domain
9,110,@no-prefix.com


In [10]:
# Manually set DNS to Google's public servers
dns.resolver.default_resolver = dns.resolver.Resolver(configure=False)
dns.resolver.default_resolver.nameservers = ['8.8.8.8', '8.8.4.4']

In [11]:
# Assuming you already have df that contain column label as 'email'
test_email = df['email'].iloc[0] # Pick the first row
print(f"Testing: '{test_email}'")

try:
    v = validate_email(test_email, check_deliverability=True)
    print("Success!")
except EmailNotValidError as e:
    print(f"Failure Reason: {e}")

Testing: 'khalid.pro@example.com'
Failure Reason: The domain name example.com does not accept email.


Now we know that the email_validator library work as attended, we will create domain_cache, so we dont repeat process for the repeating domain for each checking

In [17]:
domain_cache = {}

def validate_domain_with_cache(email):
    if pd.isna(email) or "@" not in str(email):
        return False

    # Extract the domain (e.g., 'gmail.com')
    domain = email.split('@')[-1].lower().strip()

    # Check if we already know if this domain is valid
    if domain in domain_cache:
        return domain_cache[domain]

    try:
        # Perform the actual network DNS check
        validate_email(email, check_deliverability=True)
        domain_cache[domain] = True
        return True
    except (EmailNotValidError, ValueError):
        domain_cache[domain] = False
        return False

In [18]:
# 1. Clean the data first (critical for accuracy)
df['email'] = df['email'].astype(str).str.strip().str.lower()

# 2. Run the validation
df['is_valid_domain'] = df['email'].apply(validate_domain_with_cache)

# 3. See how many are valid vs invalid
print(df['is_valid_domain'].value_counts())

is_valid_domain
False    13
True      7
Name: count, dtype: int64


In [19]:
# Create a new DF with only verified domains
valid_email_domain = df[df['is_valid_domain'] == True].copy()

# See the 'fake' or broken domains
invalid_email_domain = df[df['is_valid_domain'] == False].copy()

To have a better understanding on what domain is available for invalid and valid email domain

In [20]:
valid_email_domain['domain'] = valid_email_domain['email'].str.split('@').str[1]
invalid_email_domain['domain'] = invalid_email_domain['email'].str.split('@').str[1]

In [21]:
# 1. Generate counts for both categories
valid_counts = valid_email_domain['domain'].value_counts().reset_index()
valid_counts.columns = ['Domain', 'Count']

invalid_counts = invalid_email_domain['domain'].value_counts().reset_index()
invalid_counts.columns = ['Domain', 'Count']

# 2. Display using Tabulate format (Markdown)
print("### VALID DOMAINS DISTRIBUTION ###")
print(valid_counts.to_markdown(index=False))

print("\n" + "="*40 + "\n")

print("### INVALID DOMAINS DISTRIBUTION ###")
print(invalid_counts.to_markdown(index=False))

### VALID DOMAINS DISTRIBUTION ###
| Domain         |   Count |
|:---------------|--------:|
| gmail.com      |       2 |
| company.co.uk  |       1 |
| sub.domain.org |       1 |
| world.net      |       1 |
| provider.biz   |       1 |
| website.io     |       1 |


### INVALID DOMAINS DISTRIBUTION ###
| Domain          |   Count |
|:----------------|--------:|
| email.com       |       3 |
| example.com     |       2 |
| format          |       1 |
| domain          |       1 |
| no-prefix.com   |       1 |
| domain.com      |       1 |
| 123.123.123.123 |       1 |
| edu.gov         |       1 |
| test.com        |       1 |
